# 08 — Holdout evaluation: all tuning models (rounds 1–4)

**Summary:** Evaluate every registry row with `tuning_stage` in `round1`–`round4` on the **holdout** set from notebook **03** (labeled `id` + target SVG). Predictions are **raw** decode; postprocessing for display/submission uses **`POSTPROCESS_METHOD`** (`src.eval.postprocess_presets`).

**Recommended run order:** 03 → 04 → 05 → **08** (this notebook) → 06 → 07 → **09** (post-training holdout eval) → 10–12.

**Workflow (same idea as legacy notebook 09):**
1. Run setup cells to build `HOLDOUT_MODELS` and `_INFER_COMMON`.
2. **Either** run **one cell** to generate for **all** models, **or** run **individual generation cells** per model slot (skips automatically if that slot has no model).
3. Run **display cells** for each model index you generated (after its generation cell).

**Disk cache:** Each model folder under `EVAL_ROOT/<model_id>/` stores `predictions_raw.csv`, `predictions_post_<method>.csv`, and `eval_run_manifest.json`. If you re-run with the same holdout split, base model, adapter path, `MAX_NEW_TOKENS`, and `POSTPROCESS_METHOD`, predictions are **reloaded** from disk (no GPU regen). Set `FORCE_REGENERATE_HOLDOUT_PREDICTIONS = True` to ignore the manifest.

**Profile:** Set `RUN_PROFILE_ID` to match notebook **03**; splits, registry, and evals live under `outputs/workflow_runs/<RUN_PROFILE_ID>/`.

**Parameters:** Adjust `RUN_PROFILE_ID`, `MODEL_IDS` (empty = all round1–4 rows; else filter, **in list order**), `POSTPROCESS_METHOD`, `FORCE_REGENERATE_HOLDOUT_PREDICTIONS`, bucket sampling, and `MAX_MODEL_SLOTS` (must be ≥ `len(HOLDOUT_MODELS)`; extra slots no-op).

#### Colab / install

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Imports, paths, and parameters

In [2]:
!pip -q install transformers peft tqdm accelerate bitsandbytes cairosvg pillow matplotlib lxml pandas

In [3]:
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID =  'run_1'  #'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
EVAL_ROOT = WORKFLOW_ROOT / 'evaluations' / 'holdout_tuning'

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
MODEL_IDS = []  # empty: all tuning-stage models; else subset in this order
MAX_NEW_TOKENS = 512
POSTPROCESS_METHOD = 'current_default_sanitizer'

SEED = 42
USE_PERCENTILE_BUCKETS = True
PERCENTILE_BUCKETS = [
    ('0-10 percentile', 0.00, 0.10),
    ('40-60 percentile', 0.40, 0.60),
    ('80-100 percentile', 0.80, 1.00),
]
N_SAMPLES_PER_BUCKET = 10
N_TEXT_ROWS = 10
N_RENDER_ROWS = 10
PREVIEW_CHARS = 8000
SHOW_RAW_METRICS = True
# True: always run raw generation; False: reuse disk cache when eval_run_manifest.json matches
FORCE_REGENERATE_HOLDOUT_PREDICTIONS = False

# Number of per-model cell pairs below; increase if you have more registry rows than slots
MAX_MODEL_SLOTS = 16

print('cuda:', torch.cuda.is_available())
print('POSTPROCESS_METHOD:', POSTPROCESS_METHOD)
print('FORCE_REGENERATE_HOLDOUT_PREDICTIONS:', FORCE_REGENERATE_HOLDOUT_PREDICTIONS)
print('MAX_MODEL_SLOTS:', MAX_MODEL_SLOTS)

cuda: True
POSTPROCESS_METHOD: current_default_sanitizer
FORCE_REGENERATE_HOLDOUT_PREDICTIONS: False
MAX_MODEL_SLOTS: 16


#### Discover registry models and build session

Builds `HOLDOUT_MODELS` (ordered list), `SESSION` (in-memory display cache), and `_INFER_COMMON` kwargs shared by batch and per-slot generation. On-disk prediction reuse uses `eval_run_manifest.json` (see summary above).

In [4]:
from src.eval.holdout_tuning_notebook import (
    HoldoutTuningSession,
    build_holdout_models_list,
    prepare_holdout_and_columns,
)
from src.eval.postprocess_presets import POSTPROCESS_METHODS

SESSION = HoldoutTuningSession()
if MODEL_IDS:
    HOLDOUT_MODELS = build_holdout_models_list(
        PROJECT_DIR, workflow_root=WORKFLOW_ROOT, model_ids=MODEL_IDS
    )
else:
    HOLDOUT_MODELS = build_holdout_models_list(PROJECT_DIR, workflow_root=WORKFLOW_ROOT)
holdout, PROMPT_COL, SVG_COL = prepare_holdout_and_columns(WORKFLOW_ROOT)
RANKED_PATH = PROCESSED_DIR / 'train_ranked.csv'

if POSTPROCESS_METHOD not in POSTPROCESS_METHODS:
    raise ValueError(f'Unknown POSTPROCESS_METHOD. Pick one of: {sorted(POSTPROCESS_METHODS)}')

_INFER_COMMON = dict(
    holdout_models=HOLDOUT_MODELS,
    holdout=holdout,
    prompt_col=PROMPT_COL,
    svg_col=SVG_COL,
    base_model_id=MODEL_ID,
    max_new_tokens=MAX_NEW_TOKENS,
    postprocess_method=POSTPROCESS_METHOD,
    eval_root=EVAL_ROOT,
    ranked_path=RANKED_PATH,
    use_percentile_buckets=USE_PERCENTILE_BUCKETS,
    percentile_buckets=PERCENTILE_BUCKETS,
    n_samples_per_bucket=N_SAMPLES_PER_BUCKET,
    seed=SEED,
    show_raw_metrics=SHOW_RAW_METRICS,
    outputs_dir=WORKFLOW_ROOT,
    force_regenerate=FORCE_REGENERATE_HOLDOUT_PREDICTIONS,
    project_root=PROJECT_DIR,
    workflow_root=WORKFLOW_ROOT,
    eval_kind='holdout_tuning',
)

print('Registry models (this stage filter):', len(HOLDOUT_MODELS))
for rec in HOLDOUT_MODELS:
    print(f"  [{rec['model_index']}] {rec['model_id']} | {rec['tuning_stage']}")
if len(HOLDOUT_MODELS) > MAX_MODEL_SLOTS:
    print('WARNING: len(HOLDOUT_MODELS) > MAX_MODEL_SLOTS — increase MAX_MODEL_SLOTS in the parameters cell.')

Registry models (this stage filter): 16
  [1] c00eed730559486c | round1
  [2] 631bfea447974924 | round1
  [3] f9cfe3de45914b51 | round1
  [4] afd55604e6b647cb | round1
  [5] 35aff66e4a4d49b7 | round1
  [6] a3b2c51c4f544237 | round1
  [7] 1c3f07c240d04d1c | round1
  [8] f590f8aefc614608 | round1
  [9] e87a05edb1aa4da1 | round2
  [10] cd09b0b598b04c8e | round2
  [11] 895df98c4cff4892 | round2
  [12] 1c028d1d5dd34a1d | round3
  [13] 067768ef650c49da | round3
  [14] b90986cfef10424f | round4
  [15] eafb41fbcdc84ab7 | round4
  [16] 5d4370e7697140ba | round4


#### Generate predictions — **all models at once**

Clears the **in-memory** session only, then runs inference for every model (uses disk cache when manifest matches unless `FORCE_REGENERATE_HOLDOUT_PREDICTIONS` is True). Skip this cell if you only want per-slot generation below.

In [5]:
SESSION.clear()
SESSION.run_all_models_inference(**_INFER_COMMON)
print('Saved under:', EVAL_ROOT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Holdout SVG generation:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

#### Generate predictions — **one model per cell**

Run only the cells for the indices you need (1 … N). Slots beyond `len(HOLDOUT_MODELS)` print a skip message.

##### Model slot **1** — generation only

In [ ]:
SESSION.run_model_inference(1, **_INFER_COMMON)

##### Model slot **2** — generation only

In [ ]:
SESSION.run_model_inference(2, **_INFER_COMMON)

##### Model slot **3** — generation only

In [ ]:
SESSION.run_model_inference(3, **_INFER_COMMON)

##### Model slot **4** — generation only

In [ ]:
SESSION.run_model_inference(4, **_INFER_COMMON)

##### Model slot **5** — generation only

In [ ]:
SESSION.run_model_inference(5, **_INFER_COMMON)

##### Model slot **6** — generation only

In [ ]:
SESSION.run_model_inference(6, **_INFER_COMMON)

##### Model slot **7** — generation only

In [ ]:
SESSION.run_model_inference(7, **_INFER_COMMON)

##### Model slot **8** — generation only

In [ ]:
SESSION.run_model_inference(8, **_INFER_COMMON)

##### Model slot **9** — generation only

In [ ]:
SESSION.run_model_inference(9, **_INFER_COMMON)

##### Model slot **10** — generation only

In [ ]:
SESSION.run_model_inference(10, **_INFER_COMMON)

##### Model slot **11** — generation only

In [ ]:
SESSION.run_model_inference(11, **_INFER_COMMON)

##### Model slot **12** — generation only

In [ ]:
SESSION.run_model_inference(12, **_INFER_COMMON)

##### Model slot **13** — generation only

In [ ]:
SESSION.run_model_inference(13, **_INFER_COMMON)

##### Model slot **14** — generation only

In [ ]:
SESSION.run_model_inference(14, **_INFER_COMMON)

##### Model slot **15** — generation only

In [ ]:
SESSION.run_model_inference(15, **_INFER_COMMON)

##### Model slot **16** — generation only

In [ ]:
SESSION.run_model_inference(16, **_INFER_COMMON)

#### Cross-model summary table (postprocessed metrics)

In [ ]:
SESSION.refresh_summary_table()

#### Display comparisons — **one section per model slot**

Run the display cell for index *k* **after** you have run generation for that index. Each section shows raw vs target (metrics, text, render) then postprocessed vs target (metrics, text, render).

##### Model slot **1** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    1,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **2** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    2,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **3** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    3,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **4** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    4,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **5** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    5,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **6** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    6,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **7** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    7,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **8** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    8,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **9** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    9,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **10** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    10,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **11** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    11,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **12** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    12,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **13** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    13,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **14** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    14,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **15** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    15,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)

##### Model slot **16** — raw + postprocessed displays

In [ ]:
SESSION.display_all_comparisons_for_model(
    16,
    n_text_rows=N_TEXT_ROWS,
    n_render_rows=N_RENDER_ROWS,
    preview_chars=PREVIEW_CHARS,
)